# Heart Disease Prediction
**Steps covered:**

1. Import Libraries
2. Load Dataset
3. Understand the Dataset
4. Creating the Target Variable
5. Data Cleaning
6. Train/Test Split
7. Feature Preprocessing (Scaling & Encoding)
8. Saving Processed Data & Preprocessors
9. Train Multiple Models & Compare
10. Best Model Selection
11. Model Evaluation (Confusion Matrix, ROC & Accuracy)
12. Final Results Summary

## 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from pathlib import Path
import warnings

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_curve, auc, ConfusionMatrixDisplay
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore")

In [ ]:
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB

## 2: Load Dataset

In [ ]:
data_path = "../data/raw/heart_disease_uci.csv"
df = pd.read_csv(data_path)

print(df.shape)
df.head()

## 3: Understand the Dataset

In [ ]:
df.info()

In [ ]:
print(df.columns.tolist())

In [ ]:
print(df.isnull().sum())

## 4: Creating the Target Variable

In [ ]:
# num ranges 0-4 in the raw data, turn it into a binary target
df["target"] = (df["num"] > 0).astype(int)
print(df["target"].value_counts())

In [ ]:
target_counts = df["target"].value_counts()
labels = ["No Disease", "Disease"]

plt.figure(figsize=(7, 5))
bars = plt.bar(labels, target_counts.values)
plt.title("Heart Disease Distribution")
plt.xlabel("Disease Status")
plt.ylabel("Number of Patients")

for bar in bars:
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
              str(int(bar.get_height())), ha="center", va="bottom")

plt.tight_layout()
plt.show()

## 5: Data Cleaning

In [ ]:
df = df.drop(["num", "id", "dataset"], axis=1)
print(df.columns.tolist())

## 6: Train/Test Split

In [ ]:
X = df.drop("target", axis=1)
y = df["target"]

print(X.shape, y.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)

## 7: Feature Preprocessing (Scaling & Encoding)

In [ ]:
num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X_train.select_dtypes(include=["object", "category", "bool"]).columns

print(list(num_cols))
print(list(cat_cols))

In [ ]:
for col in num_cols:
    X_train[col] = X_train[col].fillna(X_train[col].median())
    X_test[col] = X_test[col].fillna(X_train[col].median())

for col in cat_cols:
    X_train[col] = X_train[col].fillna(X_train[col].mode()[0])
    X_test[col] = X_test[col].fillna(X_train[col].mode()[0])

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train[num_cols])
X_test_scaled = scaler.transform(X_test[num_cols])

In [ ]:
ohe = OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore")
X_train_encoded = ohe.fit_transform(X_train[cat_cols])
X_test_encoded = ohe.transform(X_test[cat_cols])

In [ ]:
X_train_final = np.hstack((X_train_scaled, X_train_encoded))
X_test_final = np.hstack((X_test_scaled, X_test_encoded))

print(X_train_final.shape, X_test_final.shape)

## 8: Saving Processed Data & Preprocessors

In [ ]:
Path("../data/processed").mkdir(parents=True, exist_ok=True)
np.save("../data/processed/X_train_final.npy", X_train_final)
np.save("../data/processed/X_test_final.npy", X_test_final)

In [ ]:
Path("../models").mkdir(parents=True, exist_ok=True)
joblib.dump(scaler, "../models/scaler.pkl")
joblib.dump(ohe, "../models/ohe.pkl")

## 9: Train Multiple Models & Compare

In [ ]:
logistic_model = LogisticRegression(max_iter=1000, random_state=42)
decision_tree_model = DecisionTreeClassifier(max_depth=5, random_state=42)
random_forest_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)

models = {
    "Logistic Regression": logistic_model,
    "Decision Tree": decision_tree_model,
    "Random Forest": random_forest_model
}

In [ ]:
svm_model = SVC(kernel="rbf", probability=True, random_state=42)
knn_model = KNeighborsClassifier(n_neighbors=5)
gradient_boosting_model = GradientBoostingClassifier(
    n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42
)
adaboost_model = AdaBoostClassifier(n_estimators=100, random_state=42)
naive_bayes_model = GaussianNB()

models.update({
    "SVM": svm_model,
    "K-Nearest Neighbors": knn_model,
    "Gradient Boosting": gradient_boosting_model,
    "AdaBoost": adaboost_model,
    "Naive Bayes": naive_bayes_model
})

In [ ]:
def evaluate_model(model, model_name):
    y_pred = model.predict(X_test_final)
    y_proba = model.predict_proba(X_test_final)[:, 1]

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)

    print("\n" + model_name)
    print("Accuracy :", round(accuracy, 4))
    print("Precision:", round(precision, 4))
    print("Recall   :", round(recall, 4))
    print("F1 Score :", round(f1, 4))
    print("ROC-AUC  :", round(roc_auc, 4))

    return {
        "Accuracy": accuracy, "Precision": precision, "Recall": recall,
        "F1": f1, "ROC_AUC": roc_auc, "fpr": fpr, "tpr": tpr
    }

In [ ]:
metrics_results = {}
trained_models = {}
roc_data = {}

for model_name, model in models.items():
    model.fit(X_train_final, y_train)
    result = evaluate_model(model, model_name)

    metrics_results[model_name] = {
        "Accuracy": result["Accuracy"], "Precision": result["Precision"],
        "Recall": result["Recall"], "F1": result["F1"], "ROC_AUC": result["ROC_AUC"]
    }
    roc_data[model_name] = {"fpr": result["fpr"], "tpr": result["tpr"], "auc": result["ROC_AUC"]}
    trained_models[model_name] = model

In [ ]:
results_df = pd.DataFrame(metrics_results).T
results_df = results_df.sort_values(by="F1", ascending=False)
print(results_df)

## 10: Best Model Selection

In [ ]:
best_model_name = results_df["F1"].idxmax()
best_model = trained_models[best_model_name]

print("Best Model:", best_model_name)
print(results_df.loc[best_model_name])

In [ ]:
joblib.dump(best_model, "../models/best_model.pkl")

## 11: Model Evaluation (Confusion Matrix, ROC & Accuracy)

In [ ]:
best_predictions = best_model.predict(X_test_final)

ConfusionMatrixDisplay.from_predictions(
    y_test, best_predictions, display_labels=["No Disease", "Disease"]
)
plt.title(f"Confusion Matrix - {best_model_name}")
plt.tight_layout()
plt.show()

In [ ]:
Path("../outputs").mkdir(parents=True, exist_ok=True)

mismatched = X_test.copy()
mismatched["actual"] = y_test
mismatched["predicted"] = best_predictions

false_negatives = mismatched[(mismatched["actual"] == 1) & (mismatched["predicted"] == 0)]
false_positives = mismatched[(mismatched["actual"] == 0) & (mismatched["predicted"] == 1)]

false_negatives.to_csv("../outputs/false_negatives.csv", index=False)
false_positives.to_csv("../outputs/false_positives.csv", index=False)

In [ ]:
plt.figure(figsize=(9, 6))
for model_name, data in roc_data.items():
    plt.plot(data["fpr"], data["tpr"], label=f"{model_name} (AUC = {data['auc']:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", label="Random Classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC-AUC Comparison of ML Models")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(results_df.index, results_df["Accuracy"])
plt.title("Accuracy Comparison of ML Models")
plt.xlabel("Machine Learning Model")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
results_df.to_csv("../outputs/model_comparison.csv")

## 12: Final Results Summary

In [ ]:
print("=" * 50)
print("FINAL MODEL RESULT")
print("=" * 50)
print("Best Model:", best_model_name)
print("Accuracy:", f"{results_df.loc[best_model_name, 'Accuracy'] * 100:.2f}%")
print("Precision:", f"{results_df.loc[best_model_name, 'Precision'] * 100:.2f}%")
print("Recall:", f"{results_df.loc[best_model_name, 'Recall'] * 100:.2f}%")
print("F1 Score:", f"{results_df.loc[best_model_name, 'F1'] * 100:.2f}%")
print("ROC-AUC:", f"{results_df.loc[best_model_name, 'ROC_AUC'] * 100:.2f}%")